In [2]:
# Author: Mengkun Tian
# Script description:
# Convert the image from dm3,dm4, tif, png, jpg, jpeg to png (tif, jpg and jpeg are also allowed) with designated output format

import hyperspy.api as hs
import numpy as np
import tkinter as tk
from tkinter import filedialog,messagebox
import secrets
from PIL import Image
from pathlib import Path
import shutil

class ImageConverter:
    def __init__(self,select_input_dir=True,select_output_dir = True, output_format = 'png', output_size = (512,512)):
        """
        select_input_dir: True  -> user chooses input directory via dialog
                     False -> use current directory ('.') as input
        select_outputdir: True  -> user chooses output directory via dialog / overwrite prompt
                      False -> auto: <input_parent>/converted_images/<input_dir_name>
        """
        self.supported_input_formats = ('dm3','dm4','tif','png','jpg','jpeg','gif')
        self.supported_output_formats = ('tif','png','jpg','jpeg','gif')
        self.output_format = output_format.lower()
        if self.output_format not in self.supported_output_formats:
            raise ValueError(f"Output format {self.output_format} is not supported. Supported formats include 'tif','png','jpg','jpeg.'")
        self.output_size = output_size
        self.select_input_dir = select_input_dir
        self.select_output_dir = select_output_dir
        self.input_path: Path | None = None   # directory input
        self.output_path: Path | None = None  # directory output
        self.input_data= None # input_data and a flag. True if input_data is given. Output image will created at the same directory as the input data ending with '_resized'.
        self.image_registration = {} # register original data: the image names, pixel size (if dm3 or dm4 data), dimension. and register the image after conversion: the image names, pixel size (if dm3 or dm4 data), dimensions. This will output as cvs file for every update of image. If the input dimension, or other information is unknown, it can allow user to manually input the information. This will be helpful for future image analysis and processing. Note everytime when csv be updated and overwritten, it will pop out a msg box. Also, the mannual input part will not be overwrite automatically unless we click update button when change happened in the mannual input part.

    def _set_input_dir(self) -> Path:
        dialog_root = tk.Tk()
        dialog_root.withdraw()
        selected_dir = filedialog.askdirectory(title = 'Please choose the input directory.',initialdir = '.')
        dialog_root.destroy()
        # if no directory selected
        if not selected_dir:
            raise RuntimeError('Please select the input directory.')
        return Path(selected_dir).resolve()

    def _set_output_dir(self)-> Path:
        """
        Working directory is the input directory
        Default_dir is the working directory's parent / "converted_images"/ working directory
        If default directory not exist, system use the current directory's parent/ "converted_images" as output directory/current directory name.
        If default directory exist, ask user decide 1. ok: override the old existing directory; 2. no: choose another directory
        """
        if self.input_path is None:
            print('Input path is not selected. Automatically select the current directory as input.')
            self.input_path = Path('.')
        input_parent_dir = self.input_path.parent
        input_dir_name = self.input_path.name
        default_output_dir = input_parent_dir / "converted_images"/input_dir_name
        if default_output_dir.exists() and any(default_output_dir.iterdir()): 
            dialog_root =tk.Tk()
            dialog_root.withdraw()
            overwrite_output = messagebox.askyesno(
                title = "Output folder exists", 
                message = f"The folder:\n{default_output_dir}\n"
                f"already exists and is not empty.\n\n"
                f"Click 'Yes' to overwrite / reuse it,\n"
                f"or 'No' to choose a different output folder."
            )
            if overwrite_output:
                shutil.rmtree(default_output_dir)#https://docs.python.org/3/library/shutil.html
                default_output_dir.mkdir(parents=True,exist_ok = True)#https://docs.python.org/3/library/pathlib.html
                selected_dir = default_output_dir
            else:
                selected_dir = filedialog.askdirectory(initialdir='.',title = "Please choose another folder as output directory")
            dialog_root.destroy()
        else:
            default_output_dir.mkdir(parents=True,exist_ok = True)
            selected_dir = default_output_dir
        return Path(selected_dir).resolve()
    def 
    
    def image_loader(self):
        """
        Images can be loaded as single, multiple images in one folder, or in a directory or a direcoty with multiple subdirectories;
        User needs to select the input path;
        1. If the image(s) directly is(are) loaded by hyperspy, meaning input_data is not None, the output path will be directly selected to the current directory. 
        2. If a directory is selected, the output directory will create a folder called converted_images. 
        The file structures are given below at different sceinarios:
        1. input_data is not None:
            input dir: 
            dir_A:
               data1, data2, data3....dataN
            output dir:
            dir_A (same directory as input, no new directory is created):
               data1, data2, data3....dataN, data1_resized, data2_resized, data3_resized....dataN_resized
                where data1_resized, data2_resized, data3_resized....dataN_resized are the images after resized and transformed
        2. input_data is None:
            a. if a single directory is selected:
                input dir:
                dir_A(sub_folder)
                    data1, data2, data3....dataM
                output dir:
                dir_A(sub_folder)
                    data1, data2, data3....dataM
                converted_image:
                    dir_A(sub_folder)
                        data1, data2, data3....dataM

            b. if a directory with multiple subdirectory is selected:
                input dir:
                dir_A (parent,user selected) 
                    dir_B(sub_folder)
                        data1, data2, data3....dataM
                    dir_C (sub folders)
                        data1, data2, data3....dataN
                output dir:
                dir_A (parent) 
                    dir_B(sub_folder)
                        data1, data2, data3....dataM
                    dir_C (sub folders)
                        data1, data2, data3....dataN
                converted_image:
                    dir_A (parent) 
                        dir_B(sub_folder,resized and transformed)
                            data1, data2, data3....dataM
                        dir_C (sub_folder,resized and transformed)
                            data1, data2, data3....dataN
        """
        # image contain is a list that has the data structure (path,hs_image_object)
        loaded_images: list[tuple[Path,object]] = []

        if self.select_input_dir:
            self.input_path = self._set_input_dir()
        else:
            self.input_path = Path('.').resolve()
        
        if self.input_data is not None: #single or multple data are loaded, no output path need to be assigned
            loaded_signal = hs.load(self.input_data)
            #handle if multiple images are loaded:
            if len(loaded_signal) ==1:
                file_stem = Path(loaded_signal.metadata.General.original_filename).stem
                image_path = self.input_path/ file_stem
                loaded_images.append((image_path,loaded_signal))
            else:
                for image_item in range(len(loaded_signal)):
                    sub_signal = loaded_signal[image_item]
                    file_stem = Path(sub_signal.metadata.General.original_filename).stem
                    image_path = self.input_path/ file_stem
                    loaded_images.append((image_path,sub_signal))
        else: # direcotry is selected, output path is designated
            #First, set the output directory;
            if self.select_output_dir:
                self.output_path = self._set_output_dir()
            else:
                input_parent_dir = self.input_path.parent
                input_dir_name = self.input_path.name
                default_output_dir = input_parent_dir / "converted_images" / input_dir_name
                default_output_dir.mkdir(parents=True, exist_ok=True)
                self.output_path = default_output_dir
            #Second, append the (file, signal)
            #for single directory with no subdirectory, use is_file to check
            for file_path in self.input_path.iterdir():
                if file_path.is_file() and file_path.suffix.lower().lstrip(".") in self.supported_input_formats: 
                    print('f = ',file_path)
                    loaded_signal = hs.load(file_path)
                    loaded_images.append((file_path,loaded_signal))
                elif file_path.is_dir():
                    for sub_file_path in file_path.iterdir():
                        if sub_file_path.is_file() and sub_file_path.suffix.lower().lstrip(".") in self.supported_input_formats: 
                            loaded_signal = hs.load(sub_file_path)
                            loaded_images.append((sub_file_path,loaded_signal))
        return loaded_images
    
    def convert_to_image(self,input_data=None,keep_original_resolution = False,roi=None):
        '''
        Params:
        input_data: the currently loaded image data
        keep_originalresolution: If True, it will force to convert the image with same original resolution by divide the image into multiple regions; 
        each region keep the same resolution as original one. Work only when the original image's resolution is the integer times of converted_image's.
        ROI: interactly select a ROI of an image to export.
        '''
        self.input_data = input_data
        loaded_images = self.image_loader()
        for image_item in loaded_images:
            source_path, image_signal = image_item
            image_data = np.array(image_signal.data)
            dtype_fields =image_data.dtype.fields
            #Handle the r,g,b cases
            if dtype_fields is not None:
                if all (channel_name in dtype_fields for channel_name in ('r','g','b')):
                    red_channel = image_data['R'].astype(float)
                    green_channel = image_data['G'].astype(float)
                    blue_channel = image_data['B'].astype(float)
                    image_data = 0.299 * red_channel + 0.587 * green_channel + 0.114 * blue_channel
            image_data =image_data.astype(float)
            #normalized data
            data_minimum = image_data.min()
            data_maximum = image_data.max()
            image_data -=data_minimum
            if data_maximum>0:
                image_data /=data_maximum
            image_data = (255*image_data).astype(np.uint8)
            #resize
            image = Image.fromarray(image_data)
            image = image.resize(size=self.output_size,resample=Image.BILINEAR)
            if self.input_data is not None:
                destination_dir = source_path.parent
                file_stem = source_path.stem
                output_stem = f'{file_stem}_resized'
                if keep_original_resolution:
                    original_height,original_width = np.shape(image_data)
                    target_height,target_width = self.output_size
                    # only support 'square' image, iterate to save all the patch image
                    # format:
                    #image_name_patchnumber.{format}
                    if target_height == target_width and original_height == original_width and original_height%target_height ==0:
                        destination_dir = source_path.parent
                        patch_output_dir = destination_dir/output_stem
                        patch_output_dir.mkdir(parents=True,exist_ok=True) #store patch of image
                        num_patches_per_axis = original_height//target_height
                        patch_index = 0
                        for patch_row in range(num_patches_per_axis):
                            for patch_col in range(num_patches_per_axis):
                                image = image_data[target_height*patch_row:target_height*(patch_row+1),target_height*patch_col:target_height*(patch_col+1)]
                                image = Image.fromarray(image)
                                unique_stem = secrets.token_hex(5)
                                patch_file_path =  patch_output_dir/f'{unique_stem}.{self.output_format}'
                                image.save(patch_file_path)
                                print(f"[OK] {source_path} -> {patch_file_path}")
                                patch_index +=1
                        continue 
                    
            else:
                #single directory case, path is the file, path.parent is the directory
                if source_path.parent == self.input_path:
                    destination_dir = self.output_path
                # directory with sub case, path is the file, path.parent is the directory
                else:
                    folder_name = source_path.parent.name
                    destination_dir = self.output_path/folder_name
                destination_dir.mkdir(parents=True,exist_ok=True)
                output_stem = secrets.token_hex(5)
            output_file_path = destination_dir/f'{output_stem}.{self.output_format}'
            image.save(output_file_path)
            print(f"[OK] {source_path} -> {output_file_path}")    

"""
Example use:
image = image_converter()
input_data = '*.jpg'
image.convert_to_image(input_data)
"""

"\nExample use:\nimage = image_converter()\ninput_data = '*.jpg'\nimage.convert_to_image(input_data)\n"

In [10]:
filename = "LF003-BROKEN_5.00kV_0.17nA_33.7nm_59_ETD_2000×.tif"
data = hs.load(filename)
data.metadata

├── Acquisition_instrument
│   └── SEM
│       ├── Stage
│       │   ├── rotation = 3.39632e-05
│       │   ├── tilt = 0
│       │   ├── x = -0.00616933
│       │   ├── y = 0.0072515
│       │   └── z = 0.0315001
│       ├── beam_current = 0.171875
│       ├── beam_energy = 5.0
│       ├── dwell_time = 3e-07
│       ├── microscope = Helios 5 CX
│       └── working_distance = 4.5184500000000005
├── General
│   ├── FileIO
│   │   └── 0
│   │       ├── hyperspy_version = 2.3.0
│   │       ├── io_plugin = rsciio.tiff
│   │       ├── operation = load
│   │       └── timestamp = 2026-05-16T19:06:18.305501-04:00
│   ├── authors = Supervisor
│   ├── date = 2025-03-14
│   ├── original_filename = LF003-BROKEN_5.00kV_0.17nA_33.7nm_59_ETD_2000×.tif
│   ├── time = 15:51:59
│   └── title = 
└── Signal
    └── signal_type =